<a href="https://colab.research.google.com/github/yvrgroupsheets/helpers/blob/main/Lots_Reconcilation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# --- CONFIGURATION ---
TOLERANCE = 50.0  # Ignore differences less than 50kg

In [4]:
# 1. LOAD DATA
# Ensure all Lot columns are strings to avoid leading zero issues
tally = pd.read_excel('/content/STOCK DUMP 24.02.2026.xlsx', sheet_name='Sheet1').rename(columns=lambda x: str(x).strip())
phys = pd.read_excel('/content/Cold Storage Stock As on 24-02-2026 lotwise.xlsx', sheet_name='AsOnQuantity').rename(columns=lambda x: str(x).strip())
moves = pd.read_excel('/content/LOCAL_MOVEMENT__INVOICE 14-02-2026.xlsx', sheet_name='DOCUMENT WISE', header=1).rename(columns=lambda x: str(x).strip())
typo_map = pd.read_excel('/content/All seed NA-srinu.xlsx', sheet_name='Total').rename(columns=lambda x: str(x).strip())
packing_plan = pd.read_excel('/content/LOCAL MOVEMENTS (10 TO 14-03-2026).xlsx', sheet_name='Sheet1', header=1).rename(columns=lambda x: str(x).strip())

print("Data Loading Complete!")
print("Tally columns:", tally.columns)
print("Tally:", tally.shape)
print("Physical columns:", phys.columns)
print("Physical:", phys.shape)
print("Moves columns:", moves.columns)
print("Moves:", moves.shape)
print("Typo Map columns:", typo_map.columns)
print("Typo Map:", typo_map.shape)
print("Packing Plan columns:", packing_plan.columns)
print("Packing Plan:", packing_plan.shape)


/usr/local/lib/python3.12/dist-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: HSPL!$A:$G.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


Data Loading Complete!
Tally columns: Index(['Particulars', 'LOT NO', 'STAGE', 'Quantity', 'COM'], dtype='object')
Tally: (8823, 5)
Physical columns: Index(['Location', 'Date', 'Gubba Memo No', 'Customer DC No', 'Company',
       'Product', 'Hybrid', 'Lot', 'Chamber', 'Rack/Bay', 'Handling Quantity',
       'Quantity', 'Vehicle No'],
      dtype='object')
Physical: (4860, 13)
Moves columns: Index(['S.NO', 'DATE', 'DC NO', 'COMP', 'MOVEMENT', 'FROM', 'TO', 'CROP',
       'STAGE', 'CODE', 'LOT NO', 'PACK\n SIZE', 'NO OF\n BAGS', 'QUANTITY',
       'ISSUE QTY', 'TOTAL \n BALANCE', 'TRANSPORT', 'VEHICLE', 'REMARKS',
       'Unnamed: 19'],
      dtype='object')
Moves: (4873, 20)
Typo Map columns: Index(['Product', 'Hybrid', 'Lot', 'LOT DIFF', 'Gubba Quantity', 'Tally Qty',
       'diff', 'Gubba Location', 'Tally Location', 'Chamber', 'Rack/Bay',
       'Handling Quantity', 'Other Location', 'Remarks'],
      dtype='object')
Typo Map: (133, 14)
Packing Plan columns: Index(['S.NO', 'DATE', 'D

In [12]:
# 2. APPLY TYPO MAPPING
# Convert typo_map into a dictionary {Physical_Name: Tally_Name}
mapping_dict = dict(zip(typo_map['Lot'].astype(str), typo_map['LOT DIFF'].astype(str)))

def clean_lot(lot):
    lot_str = str(lot).strip()
    return mapping_dict.get(lot_str, lot_str)

# Clean lot names across all dataframes
tally['lot'] = tally['LOT NO'].astype(str).str.strip()
phys['lot'] = phys['Lot'].apply(clean_lot)
moves['lot'] = moves['LOT NO'].apply(clean_lot)
packing_plan_lots = set(packing_plan['LOT NO'].astype(str).str.strip())

print("Typo Mapping Complete!")
print("moves Columns:", moves.columns)
print("moves lots", moves['lot'].count())


Typo Mapping Complete!
moves Columns: Index(['S.NO', 'DATE', 'DC NO', 'COMP', 'MOVEMENT', 'FROM', 'TO', 'CROP',
       'STAGE', 'CODE', 'LOT NO', 'PACK\n SIZE', 'NO OF\n BAGS', 'QUANTITY',
       'ISSUE QTY', 'TOTAL \n BALANCE', 'TRANSPORT', 'VEHICLE', 'REMARKS',
       'Unnamed: 19', 'lot'],
      dtype='object')
moves lots 4873


In [17]:
# 3. AGGREGATE QUANTITIES (Summing multiple vehicles/batches)
tally_grouped = tally.groupby('lot')['Quantity'].sum().reset_index().rename(columns={'Quantity': 'tally_qty'})
phys_grouped = phys.groupby('lot')['Quantity'].sum().reset_index().rename(columns={'Quantity': 'phys_qty'})
moves_grouped = moves.groupby('lot')['QUANTITY'].sum().reset_index().rename(columns={'QUANTITY': 'move_qty'})

In [18]:
# 4. MERGE ALL DATA SOURCES
# Start with an outer join to capture every lot mentioned anywhere
master = pd.merge(tally_grouped, phys_grouped, on='lot', how='outer')
master = pd.merge(master, moves_grouped, on='lot', how='outer')

# Fill NaN with 0
master[['tally_qty', 'phys_qty', 'move_qty']] = master[['tally_qty', 'phys_qty', 'move_qty']].fillna(0)

print("Data Aggregation Complete!")
print("Master columns:", master.columns)
print("Master:", master.shape)

Data Aggregation Complete!
Master columns: Index(['lot', 'tally_qty', 'phys_qty', 'move_qty'], dtype='object')
Master: (8511, 4)


In [19]:
# 5. CALCULATE LOGIC
master['total_system_qty'] = master['tally_qty'] + master['move_qty']
master['variance'] = master['phys_qty'] - master['total_system_qty']


In [20]:
# 6. ASSIGN STATUS
def get_status(row):
    if row['total_system_qty'] > 0 and row['phys_qty'] == 0:
        return 'GHOST'
    if row['total_system_qty'] == 0 and row['phys_qty'] > 0:
        return 'ABANDONED'
    if abs(row['variance']) > TOLERANCE:
        return 'QTY MISMATCH'
    return 'MATCHED'

master['status'] = master.apply(get_status, axis=1)
master['priority'] = master['lot'].apply(lambda x: 'CRITICAL' if x in packing_plan_lots else 'Normal')


In [21]:
# 7. FILTER & SORT
# Keep only discrepancies OR lots in the packing plan
final_report = master[(master['status'] != 'MATCHED') | (master['priority'] == 'CRITICAL')].copy()
final_report = final_report.sort_values(by=['priority', 'status'], ascending=[True, False])


In [22]:
# 8. EXPORT
final_report.to_excel('Recon_Report_Final.xlsx', index=False)
print("Reconciliation Complete! Download 'Recon_Report_Final.xlsx' from the file sidebar.")


Reconciliation Complete! Download 'Recon_Report_Final.xlsx' from the file sidebar.
